# Plánování radioterapie

## Zadání slovy

> V řezu pacientem leží nádor a hned vedle něj mícha, kterou nesmíme přezářit.
> Ozařovač umí vyslat úzký svazek z osmi úhlů kolem pacienta, v každém úhlu ve
> 12 příčných polohách, a u každého z těch 96 svazků se dá nastavit intenzita.
> Dávka se v tkáni sčítá.
>
> **Jak nastavit 96 intenzit, aby nádor dostal předepsaných 60 Gy a mícha
> i okolní zdravá tkáň co nejméně?**

Fyzika je celá v **matici dávek** $D$: prvek $D_{ij}$ říká, kolik dávky dostane
voxel $i$ od svazku $j$ s jednotkovou intenzitou. Je to Gaussův profil napříč
svazkem (σ = 2,6 voxelu, takže se sousední polohy překrývají) krát exponenciální
útlum s hloubkou. Rozdělení dávky v celém řezu je pak **lineární** funkce
nastavení, $\mathbf d = D\mathbf w$ — a právě proto je z toho zvládnutelná úloha.

## Formulace

Nechť $T$ je množina voxelů nádoru, $R$ voxelů míchy (rizikový orgán) a $Z$
voxelů zdravé tkáně.

$$
\begin{aligned}
\text{minimize}_{\mathbf w}\quad & 3\sum_{i \in R} (D\mathbf w)_i^2
  \;+\; \sum_{i \in Z} (D\mathbf w)_i^2 && \text{davka mimo nador}\\
\text{subject to}\quad & 60 \le (D\mathbf w)_i \le 69,\quad i \in T
  && \text{predpis do nadoru (Gy)}\\
& \mathbf w \ge \mathbf 0 && \text{intenzity nejsou zaporne}
\end{aligned}
$$

Proměnné jsou intenzity svazků $\mathbf w \in \mathbb{R}^{96}$, účelová funkce
je vážený součet čtverců dávky mimo nádor a omezení jsou lineární. Jde tedy
o **konvexní kvadratický program**: každé lokální minimum je globální a solver
vrací certifikát optimality. Váha 3 u míchy není fyzika, ale klinická preference
— je to první místo, kde se dá s modelem hrát.

## Od zadání ke kódu

| v zadání | v kódu |
|---|---|
| matice dávek $D$ | `D` — řádky voxely, sloupce svazky |
| množiny $T$, $R$, $Z$ | masky `nador`, `micha`, `zdrava` |
| $\mathbf w \ge \mathbf 0$ | `w = cp.Variable(D.shape[1], nonneg=True)` |
| $(D\mathbf w)_i$ pro $i \in T$ | `d_nador = D[nador.ravel()] @ w` |
| $3\sum_R(\cdot)^2 + \sum_Z(\cdot)^2$ | `VAHA_MICHA * cp.sum_squares(d_micha) + cp.sum_squares(d_zdrava)` |
| $60 \le (D\mathbf w)_i \le 69$ | `[d_nador >= PREDPIS, d_nador <= TOLERANCE * PREDPIS]` |

Plná verze téhle ukázky, ze které notebook vychází, je ve skriptu [`kod/ukazka_radioterapie.py`](https://github.com/tomasvicar/OMM-public/blob/master/cviceni/C1/kod/ukazka_radioterapie.py) v repozitáři předmětu.

In [ ]:
try:
    import cvxpy as cp
except ImportError:
    %pip install -q cvxpy
    import cvxpy as cp

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np

## Řez pacientem a matice dávek

Nejdřív vznikne geometrie: kruhové tělo, v něm nádor a kousek vedle mícha. Pak
se pro každý z 96 svazků spočítá, kolik dávky dá do každého voxelu — to je
matice $D$. Do optimalizace pak jde už jen ta matice, o anatomii solver nic neví.

In [ ]:
N = 64  # rozlišení řezu
yy, xx = np.mgrid[0:N, 0:N]
c = (N - 1) / 2
telo = np.hypot(xx - c, yy - c) < 30
nador = np.hypot(xx - c - 6, yy - c + 4) < 7
micha = np.hypot(xx - c + 8, yy - c - 6) < 4
zdrava = telo & ~nador & ~micha

uhel = np.deg2rad(np.arange(0, 360, 45))[:, None, None, None]  # 8 úhlů ozařovače
posun = np.linspace(-18, 18, 12)[None, :, None, None]  # 12 poloh svazku v úhlu
pricne = (xx - c) * np.cos(uhel) + (yy - c) * np.sin(uhel)  # napříč svazkem
hloubka = -(xx - c) * np.sin(uhel) + (yy - c) * np.cos(uhel)  # podél svazku
D = np.exp(-0.5 * ((pricne - posun) / 2.6) ** 2) * np.exp(-0.02 * (hloubka + 32)) * telo
D = D.reshape(-1, N * N).T  # dávka do voxelu i od svazku j s jednotkovou intenzitou
print(f"matice dávek D: {D.shape[0]} voxelů × {D.shape[1]} svazků")

## Model a řešení

In [ ]:
VAHA_MICHA = 3  # @param {type:"slider", min:0, max:100, step:1}
PREDPIS = 60  # @param {type:"slider", min:30, max:80, step:1}
TOLERANCE = 1.15  # @param {type:"slider", min:1.05, max:1.4, step:0.05}

w = cp.Variable(D.shape[1], nonneg=True)  # intenzity svazků, nezáporné
d_nador, d_micha, d_zdrava = D[nador.ravel()] @ w, D[micha.ravel()] @ w, D[zdrava.ravel()] @ w
uloha = cp.Problem(
    cp.Minimize(VAHA_MICHA * cp.sum_squares(d_micha) + cp.sum_squares(d_zdrava)),
    [d_nador >= PREDPIS, d_nador <= TOLERANCE * PREDPIS],
)
uloha.solve(solver=cp.CLARABEL)

print(f"stav řešení: {uloha.status}")
print(f"zapnutých svazků: {(w.value > 1e-3 * w.value.max()).sum()} z {D.shape[1]}")

## Kontrola, která umí selhat

Solver vrací intenzity, ne dávku — tak se dávka spočítá **znovu** z vrácených
intenzit a ověří se, že celý nádor opravdu padne do předepsaného pásma.
Srovnávacím plánem je „rovnoměrné ozáření“: všechny svazky stejně silné,
naškálované tak, aby nádor dostal tutéž minimální dávku. Ten je přípustný taky,
jen do míchy pošle násobně víc.

In [ ]:
davka = D @ w.value  # dávka přepočítaná zpět z vráceného řešení
rovno = D @ np.ones(D.shape[1])
rovno *= PREDPIS / rovno[nador.ravel()].min()  # stejné minimum v nádoru

print(f"nádor {davka[nador.ravel()].min():.1f}–{davka[nador.ravel()].max():.1f} Gy "
      f"(předpis {PREDPIS:.0f}–{TOLERANCE * PREDPIS:.0f} Gy)")
print(f"mícha max: rovnoměrně {rovno[micha.ravel()].max():.1f} Gy, "
      f"optimalizace {davka[micha.ravel()].max():.1f} Gy")

assert davka[nador.ravel()].min() >= PREDPIS - 1e-6
assert davka[nador.ravel()].max() <= TOLERANCE * PREDPIS + 1e-6
print("kontrola dosazením zpět: celý nádor je v předepsaném pásmu")

## Obrázek

Dose-volume histogram: pro každou dávku na vodorovné ose říká, kolik procent
objemu dané tkáně jí dostane aspoň tolik. Ideál je svislá čára u nádoru
a co nejvíc vlevo stlačená křivka u míchy.

In [ ]:
for maska, popis in [(nador, "nádor"), (micha, "mícha"), (zdrava, "zdravá tkáň")]:
    objem = np.linspace(0, 100, int(maska.sum()))
    cara, = plt.plot(np.sort(davka[maska.ravel()])[::-1], objem, label=popis)
    plt.plot(np.sort(rovno[maska.ravel()])[::-1], objem, "--", color=cara.get_color())
plt.xlabel("dávka [Gy]")
plt.ylabel("podíl objemu s aspoň touto dávkou [%]")
plt.title("Dose-volume histogram (plně optimalizace, čárkovaně rovnoměrně)")
plt.legend()
plt.show()

## Na co se zeptat kódu

1. Nastavte váhu míchy na 0 a pak na 100 — o kolik klesne dávka v míše a komu se
   to připíše?
2. Zúžení tolerance na 1,05 nádor zhomogenizuje. Kolik za to zaplatí mícha?
3. Co se stane, když dávku v nádoru místo předepsání *maximalizujete*?